# 第90课 · 让机器记住对话、答有出处，还会自己动手——对话式 RAG 与工具调用 Agent（ReAct）

**目标**：串联 RAG 检索 + 本地 LLM，构建能回答音频/播客内容问题的对话引擎；并在其上从零手写一个 ReAct 工具调用 Agent（Thought → Action → Observation 循环）

🔗 **Aurora 连接**：本节是 `aurora/podcast_intelligence/` 的完整交付物——`podcast_qa()` 函数将作为对话引擎的核心入口，供 L93/L94 的评估流水线和最终 demo 直接调用。

← **上一课**　[L89 · RAG 完整流水线](L89_rag_pipeline.ipynb)

> 上节课学习了 **RAG 完整流水线**：文档切片 + TF-IDF 检索 + 提示词构建 + 生成。  
> 本课将探讨 **对话式 RAG**。

## 你有没有和 ChatGPT 聊超过 10 轮，然后它忘记了你最开始说的事？

这是 LLM 的两大痛点：

1. **记忆有限**：Context 窗口有长度上限，对话越长越容易"忘事"
2. **知识截止**：LLM 不知道你的播客内容，只知道训练数据截止日期之前的世界

**对话式 RAG 同时解决这两个问题**：

```
你的问题 → [滑动窗口截断历史] → 只保留最近5轮
               ↓
         [TF-IDF检索] → 从知识库捞出最相关的3段播客文字
               ↓
         [Prompt组装] → 把历史+检索结果+问题拼成一个结构化prompt
               ↓
         [LLM生成] → 只基于上下文回答（不编造）
               ↓
         答案 + 来源标注（哪个episode第几分钟）
```

**三个工程构件**：
- `build_prompt(context_chunks, history, question)` — 组装完整prompt
- `format_sources(source_chunks)` — 来源归因格式化
- `truncate_history(history, max_turns=5)` — 滑动窗口（Sliding Window）防token爆炸

**主函数**：`podcast_qa(question, rag_index, chunks, embed_fn, llm, history)` — 把这三件事串起来的5步流水线。

## 核心直觉：LLM 是厨师，RAG 是菜谱

没有 RAG 时，LLM 像一位只凭"感觉"炒菜的厨师——可能做出精彩的菜，也可能凭空发明从未存在的配方（幻觉，Hallucination）。

有了 RAG，你给厨师一张**即时取出的菜谱**（检索到的 chunk）。他必须遵照菜谱做，不能乱加料——这就是为什么 prompt 里要写"仅基于以上内容回答，若上下文不含答案请明确说无法回答"。

**为什么把 context 放在 history 之前？**
模型先读"事实文本"再读"对话流"，能避免历史对检索结果的干扰——否则模型可能把之前聊过的内容当成新的"事实"。

**来源归因（Source Attribution）的工程价值**：
- 用户可点击时间戳跳回播客验证
- 自动评估时，Precision@k = 检索到的 k 个 chunk 中有多少包含正确答案

In [ ]:
# ── aurora.llm 导入（TF-IDF 稀疏检索） ──────────────────────────────────────
#
# L89 实现了 build_tfidf / cosine_retrieve（词频统计 × 逆文档频率 的稀疏向量）。
# 本节在此基础上提供兼容接口 build_rag_index / retrieve，并演示如何扩展到
# 稠密嵌入（Dense Embedding）路径——见下方桩实现中的 embed_fn 参数。
#
# 稀疏 vs 稠密检索对比：
#   TF-IDF（L89/L90 aurora.llm）：词频统计，无需模型，速度快，OOV 词无效
#   Dense Embedding（L90 桩）：语义相似，需要嵌入模型，泛化更好但更重
try:
    from aurora.llm import build_tfidf, cosine_retrieve

    def build_rag_index(chunks, embed_fn=None):
        """使用 aurora.llm TF-IDF 构建检索索引。
        
        返回 (tfidf_matrix, vocab, chunks) 三元组供 retrieve() 使用。
        embed_fn 参数保留但在 TF-IDF 路径中忽略。
        """
        texts = [c["text"] if isinstance(c, dict) else c for c in chunks]
        matrix, vocab = build_tfidf(texts)
        return (matrix, vocab, chunks)

    def retrieve(query, rag_index, chunks, embed_fn=None, top_k=3):
        """TF-IDF 余弦相似度检索，返回 top_k 个最相关的 chunk 字典列表。"""
        matrix, vocab, orig_chunks = rag_index
        texts = [c["text"] if isinstance(c, dict) else c for c in orig_chunks]
        results = cosine_retrieve(query, matrix, vocab, texts, top_k=top_k)
        # 把检索命中的文本映射回原始 chunk 字典（保留 source 字段）
        text_to_chunk = {}
        for ch in orig_chunks:
            t = ch["text"] if isinstance(ch, dict) else ch
            text_to_chunk[t] = ch
        return [text_to_chunk.get(doc, {"text": doc, "source": ""})
                for doc, _ in results]

    print("✅ aurora.llm 导入成功（TF-IDF 检索路径）")
except ImportError:
    import numpy as np

    def _cosine(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

    def build_rag_index(chunks, embed_fn=None):
        """稠密嵌入桩索引（aurora.llm 不可用时的后备）。
        
        返回嵌入矩阵 ndarray，形状 (n_chunks, dim)。
        注意：embed_fn 若为 None，则使用随机向量（仅供演示）。
        """
        import numpy as np
        if embed_fn is None:
            rng = np.random.default_rng(0)
            return rng.standard_normal((len(chunks), 8)).astype(np.float32)
        vecs = np.array([
            embed_fn(c["text"] if isinstance(c, dict) else c) for c in chunks
        ], dtype=np.float32)
        return vecs

    def retrieve(query, rag_index, chunks, embed_fn=None, top_k=3):
        """稠密嵌入余弦检索（桩版本）。"""
        import numpy as np
        vecs = rag_index
        if embed_fn is None:
            return chunks[:top_k]
        q_vec = embed_fn(query)
        scores = [_cosine(q_vec, v) for v in vecs]
        idxs = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        return [chunks[i] for i in idxs]

    print("⚠️  使用稠密嵌入桩（aurora.llm 未找到，请运行 make install）")

## 1. RAG Prompt 格式

RAG prompt 由四层拼接而成，顺序不能乱：

```
[SYSTEM]
你是播客助手，只基于提供的上下文回答问题，不要编造信息。

[CONTEXT]
--- 片段 1 (来源: episode_42.txt, t=00:12:30) ---
"John played lead guitar on that track..."

--- 片段 2 (来源: episode_42.txt, t=00:15:02) ---
"The band lineup included Sarah on bass..."

[HISTORY]
User: Who produced the album?
Assistant: The album was produced by Rick Rubin (来源: episode_42.txt, t=00:08:10).

[QUESTION]
Who played guitar in this episode?

仅基于以上内容回答。若上下文不含答案，请明确说"根据已有内容无法回答"。
```

关键设计决策：
- `[CONTEXT]` 在 `[HISTORY]` 之前——让模型先读事实再读对话流，减少历史对检索结果的干扰
- "仅基于以上内容回答" 是硬约束，防止模型混入预训练记忆
- 每个片段标注来源，方便后续来源归因

In [ ]:
def build_prompt(context_chunks, history, question, system_prompt=None):
    """
    把检索片段 + 对话历史 + 当前问题拼成完整 prompt 字符串。

    Parameters
    ----------
    context_chunks : list[dict]
        每个元素含 'text' 和 'source' 键。
    history : list[dict]
        每个元素含 'role'('user'/'assistant') 和 'content' 键。
    question : str
        当前用户问题。
    system_prompt : str | None
        若为 None 则使用默认系统提示。

    Returns
    -------
    str
        组装好的 prompt。
    """
    if system_prompt is None:
        system_prompt = (
            "你是播客助手，只基于提供的上下文回答问题，不要编造信息。\n"
            "若上下文不含答案，请明确说「根据已有内容无法回答」。"
        )

    parts = [f"[SYSTEM]\n{system_prompt}\n"]

    # 上下文片段
    parts.append("[CONTEXT]")
    for i, chunk in enumerate(context_chunks, 1):
        src = chunk.get("source", "未知来源")
        text = chunk.get("text", "")
        parts.append(f"--- 片段 {i} (来源: {src}) ---\n{text}")
    parts.append("")

    # 历史对话
    if history:
        parts.append("[HISTORY]")
        for turn in history:
            role = "User" if turn["role"] == "user" else "Assistant"
            parts.append(f"{role}: {turn['content']}")
        parts.append("")

    # 当前问题
    parts.append(f"[QUESTION]\n{question}\n\n仅基于以上内容回答。")

    return "\n".join(parts)


# 演示
demo_chunks = [
    {"text": "John played lead guitar on that track.", "source": "ep42.txt t=00:12:30"},
    {"text": "Sarah was on bass for the whole session.", "source": "ep42.txt t=00:15:02"},
]
demo_history = [{"role": "user", "content": "Who produced the album?"},
                {"role": "assistant", "content": "Rick Rubin (来源: ep42.txt t=00:08:10)."}]
prompt = build_prompt(demo_chunks, demo_history, "Who played guitar?")
print(prompt)

## 2. 来源归因（Source Attribution）

每次检索都应该记录是从哪个文档的哪个位置找到的。格式：

```
来源：episode_42.txt, t=00:12:30
```

实现方式：`retrieve()` 返回的每个 chunk 字典里带 `source` 字段；`podcast_qa()` 把这些 source 列表一并返回给调用方，前端/评估脚本可以直接展示或记录日志。

来源归因的价值：
- 用户可验证答案的真实性（点击时间戳跳回播客）
- 评估时可自动对比"检索来源"与"标注答案来源"，计算 Precision@k

In [ ]:
def format_sources(source_chunks):
    """
    把来源 chunk 列表格式化为可读字符串列表。

    Parameters
    ----------
    source_chunks : list[dict]
        含 'source' 键的 chunk 列表。

    Returns
    -------
    list[str]
        格式化的来源字符串列表，去重并保序。
    """
    seen = set()
    result = []
    for chunk in source_chunks:
        src = chunk.get("source", "未知来源")
        if src not in seen:
            seen.add(src)
            result.append(f"📄 {src}")
    return result


# 演示
srcs = format_sources(demo_chunks)
for s in srcs:
    print(s)
print("✅ 来源格式化正常")

## 3. 会话记忆与滑动窗口

对话历史越长，token 消耗越多，最终会超出 LLM 的 context 长度限制。解决方案：**滑动窗口**——只保留最近 `k` 轮对话。

```
history = history[-(2*k):]   # 每轮含 user + assistant 两条，共 2k 条
```

更精细的做法是按 token 数截断，而不是按轮数，但对播客 QA 场景，k=5 轮（≈1000 token）已经够了。

注意：截断历史不影响向量检索的质量——检索走的是嵌入（Embedding）空间，与 history 无关。

In [ ]:
def truncate_history(history, max_turns=5):
    """
    保留最近 max_turns 轮对话（每轮 = user + assistant 各一条）。

    Parameters
    ----------
    history : list[dict]
        对话历史，每条含 'role' 和 'content'。
    max_turns : int
        最多保留的轮数。

    Returns
    -------
    list[dict]
        截断后的历史。
    """
    max_entries = max_turns * 2   # user + assistant
    return history[-max_entries:] if len(history) > max_entries else history


# 演示：构造 8 轮历史，截断到 3 轮
long_history = []
for i in range(8):
    long_history.append({"role": "user", "content": f"问题{i+1}"})
    long_history.append({"role": "assistant", "content": f"回答{i+1}"})

short = truncate_history(long_history, max_turns=3)
print(f"原始历史条数: {len(long_history)}")
print(f"截断后条数:   {len(short)}")
print(f"最早保留轮次: {short[0]['content']}")
print("✅ 滑动窗口正常")

## 4. ✏️ 实现 `podcast_qa(question, rag_index, chunks, embed_fn, llm, history=None)`

**推理路线**：
1. `truncate_history(history, max_turns=5)` → 裁剪历史，防止 token 爆炸
2. `retrieve(question, rag_index, chunks, embed_fn, top_k=3)` → 拿到最相关的 3 个 context chunk
3. `build_prompt(context_chunks, history, question)` → 组装完整 prompt
4. `llm.generate(prompt)` → 调用本地 LLM 生成答案字符串
5. 返回 `(answer: str, sources: list[str])`，sources 由 `format_sources(context_chunks)` 生成

**参考输入输出**：

```python
question  = "Who played guitar in this episode?"
# retrieve 找到:
#   chunk_0: {"text": "John played lead guitar...", "source": "ep42.txt t=00:12:30"}
#   chunk_1: {"text": "The session featured...",   "source": "ep42.txt t=00:14:55"}
#   chunk_2: {"text": "Sarah on bass...",           "source": "ep42.txt t=00:15:02"}
# llm 生成:
answer  = "John played lead guitar in this episode."
sources = ["📄 ep42.txt t=00:12:30", "📄 ep42.txt t=00:14:55", "📄 ep42.txt t=00:15:02"]
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def podcast_qa(question, rag_index, chunks, embed_fn, llm, history=None):
    """
    播客问答主函数：检索相关片段 → 组装 prompt → LLM 生成 → 返回答案与来源。

    Parameters
    ----------
    question  : str          当前用户问题
    rag_index : np.ndarray   由 build_rag_index 生成的嵌入矩阵
    chunks    : list[dict]   原始 chunk 列表，每条含 'text' 和 'source'
    embed_fn  : callable     文本 → 向量的嵌入函数
    llm       : object       带 .generate(prompt: str) -> str 方法的 LLM 对象
    history   : list[dict]   对话历史，默认 None（空列表）

    Returns
    -------
    answer  : str
    sources : list[str]
    """
    if history is None:
        history = []

    # ✏️ TODO 步骤 1：用 truncate_history 裁剪 history（max_turns=5）
    raise NotImplementedError("TODO: 用 truncate_history 裁剪 history（max_turns=5）")

    # ✏️ TODO 步骤 2：调用 retrieve 获取 context_chunks（top_k=3）
    raise NotImplementedError("TODO: 调用 retrieve 获取 context_chunks（top_k=3）")

    # ✏️ TODO 步骤 3：调用 build_prompt 组装完整 prompt
    raise NotImplementedError("TODO: 调用 build_prompt 组装完整 prompt")

    # ✏️ TODO 步骤 4：调用 llm.generate(prompt) 得到 answer
    raise NotImplementedError("TODO: 调用 llm.generate(prompt) 得到 answer")

    # ✏️ TODO 步骤 5：调用 format_sources 并返回 (answer, sources)
    raise NotImplementedError("TODO: 调用 format_sources 生成来源列表")
    return answer, sources

In [ ]:
# ── 最小桩 LLM，用于单元测试（不调用真实模型）──────────────────────────
import numpy as np

class _StubLLM:
    """假 LLM：从 prompt 的 [CONTEXT] 段落回显检索到的片段文本，仅供检查用。

    真实 LLM 会阅读整段 prompt 并综合作答；这里用一个确定性桩：
    只提取 [CONTEXT] 段的片段正文并拼接返回，忽略 [SYSTEM]/[HISTORY]/[QUESTION]。
    这样「有 RAG」时答案必然引用检索到的 chunk 文本，便于验证流水线是否接通。
    """
    def generate(self, prompt):
        context_lines = []
        in_context = False
        for line in prompt.split("\n"):
            stripped = line.strip()
            if stripped.startswith("[CONTEXT]"):
                in_context = True
                continue
            if in_context and stripped.startswith("["):   # 到达 [HISTORY]/[QUESTION]
                break
            # 跳过 "--- 片段 N (来源: ...) ---" 分隔行，只留正文
            if in_context and stripped and not stripped.startswith("---"):
                context_lines.append(stripped)
        if context_lines:
            return " ".join(context_lines)   # 综合所有检索片段作为答案
        return "根据提供的上下文无法确定。"

def _stub_embed(text_or_chunk):
    """把文本哈希成 8 维向量（仅供测试）。"""
    text = text_or_chunk["text"] if isinstance(text_or_chunk, dict) else text_or_chunk
    rng = np.random.default_rng(abs(hash(text)) % (2**31))
    return rng.standard_normal(8).astype(np.float32)

# 构造测试数据
_test_chunks = [
    {"text": "John played lead guitar on that track.", "source": "ep42.txt t=00:12:30"},
    {"text": "Sarah was on bass for the whole session.", "source": "ep42.txt t=00:15:02"},
    {"text": "The producer was Rick Rubin.", "source": "ep42.txt t=00:08:10"},
    {"text": "Drums were handled by Mike.", "source": "ep42.txt t=00:20:00"},
]
_test_index = build_rag_index(_test_chunks, _stub_embed)
_test_llm   = _StubLLM()

try:
    answer, sources = podcast_qa(
        question   = "Who played guitar in this episode?",
        rag_index  = _test_index,
        chunks     = _test_chunks,
        embed_fn   = _stub_embed,
        llm        = _test_llm,
        history    = [],
    )
except (NotImplementedError, TypeError):
    print('⬜ 请先实现 podcast_qa() 的5个TODO步骤')
else:
    print('答案：', answer)
    print('来源：')
    for s in sources:
        print('  ', s)
    assert isinstance(answer, str) and len(answer) > 0, 'answer 应为非空字符串'
    assert isinstance(sources, list) and len(sources) > 0, 'sources 应为非空列表'
    assert all(s.startswith('📄') for s in sources), '每条来源应以 📄 开头'
    print('\n✅ podcast_qa 基本检查通过')


## 5. 参数实验：有无 RAG 的回答质量对比

**实验设置**：
- 构建 10 个模拟播客 episode 的索引（每个 episode 4 个 chunk，共 40 条）
- 测试 5 个问题，分别记录：
  - `with_rag=True`：走完整 `podcast_qa()` 流程
  - `with_rag=False`：直接把问题喂给 LLM（不检索，不注入 context）

**预期现象**：
- `top_k=3`：通常足够，增大到 5 可能引入噪声 chunk，导致答案不聚焦
- `max_turns=5`：历史超过 5 轮时截断，答案稳定；设为 0 则每轮独立，失去上下文连贯性
- 无 RAG 时：LLM 对播客特定细节（人名、时间戳、观点）往往幻觉或回答"不知道"
- 有 RAG 时：答案直接引用 chunk 文本，来源可追溯

**关键指标**：`Precision@3`（3 个检索结果中有多少含正确答案的 chunk）

In [ ]:
import numpy as np

# ── 构造 10 个 episode × 4 chunk 的合成数据集 ──────────────────────
np.random.seed(42)

episode_facts = [
    ("Alice", "piano"), ("Bob", "guitar"), ("Carol", "drums"),
    ("Dave", "bass"),   ("Eve", "violin"), ("Frank", "trumpet"),
    ("Grace", "flute"), ("Hank", "cello"), ("Iris", "harp"), ("Jack", "sitar"),
]

synth_chunks = []
for ep_idx, (name, instrument) in enumerate(episode_facts):
    ep_name = f"episode_{ep_idx+1:02d}.txt"
    synth_chunks += [
        {"text": f"{name} played {instrument} in the session.",
         "source": f"{ep_name} t=00:10:{ep_idx*3:02d}"},
        {"text": f"The host interviewed {name} about their career.",
         "source": f"{ep_name} t=00:20:{ep_idx*2:02d}"},
        {"text": f"Audience Q&A focused on {instrument} technique.",
         "source": f"{ep_name} t=00:35:00"},
        {"text": f"Closing remarks from {name}.",
         "source": f"{ep_name} t=00:55:00"},
    ]

synth_index = build_rag_index(synth_chunks, _stub_embed)

# ── 5 个测试问题（附正确答案供对比）──────────────────────────────────
test_questions = [
    ("Who plays guitar?",  "Bob"),
    ("Who plays piano?",   "Alice"),
    ("Who plays drums?",   "Carol"),
    ("Who plays violin?",  "Eve"),
    ("Who plays cello?",   "Hank"),
]

# ── 对比实验 ───────────────────────────────────────────────────────────
class _NoContextLLM:
    """模拟无 RAG 时 LLM 的行为：不看上下文，直接猜。"""
    def generate(self, prompt):
        return "我不确定，可能是某位音乐家。"   # 典型幻觉/拒答

try:
    print(f"{'问题':<30} {'期望':<10} {'RAG答案':<55} {'无RAG答案'}")
    print("-" * 120)
    n_hit = 0
    for question, expected in test_questions:
        # 有 RAG
        ans_rag, srcs = podcast_qa(question, synth_index, synth_chunks, _stub_embed, _test_llm)
        # 无 RAG（直接调用 LLM，跳过检索）
        ans_nrag = _NoContextLLM().generate(question)
        hit = expected.lower() in ans_rag.lower()
        n_hit += hit
        mark = "✅" if hit else "❌"
        print(f"{question:<30} {expected:<10} {mark} {ans_rag[:50]:<52} {ans_nrag}")
    print()
    print(f"命中率（RAG 答案含正确人名）：{n_hit}/{len(test_questions)}")
    print("观察：有 RAG 时答案直接引用 chunk 文本；无 RAG 时 LLM 无法回答播客特定细节。")
except (NotImplementedError, TypeError):
    print("⬜ 请先实现 podcast_qa() 再运行对比实验")

## ✏️ 闭卷推导检查格 — 对话式 RAG 数据流

**规则：关闭上方所有格，仅凭记忆完成。**

**题目**：`podcast_qa()` 的5步流水线，填写每步的输入→输出类型：

| 步骤 | 函数 | 输入 | 输出 |
|---|---|---|---|
| 1 | `truncate_history` | `history: list[dict]` | ___（list[dict], 最多max_turns*2条） |
| 2 | `retrieve` | `question: str, rag_index, chunks` | ___（list[dict], top_k个chunk） |
| 3 | `build_prompt` | `context_chunks, history, question` | ___（str, 完整prompt） |
| 4 | `llm.generate` | `prompt: str` | ___（str, 答案） |
| 5 | `format_sources` | `context_chunks: list[dict]` | ___（list[str], 来源列表） |

并回答：
1. 为什么 `truncate_history` 必须在 `retrieve` 之前调用？
2. `format_sources` 的去重逻辑用了什么数据结构？时间复杂度？

（在此处填写...）

In [ ]:
# 验证：podcast_qa 端到端集成（若已实现）
try:
    _q = "Who plays guitar?"
    _ans, _srcs = podcast_qa(
        question=_q,
        rag_index=_test_index,
        chunks=_test_chunks,
        embed_fn=_stub_embed,
        llm=_test_llm,
        history=[],
    )
    assert isinstance(_ans, str) and len(_ans) > 0, "answer 应为非空字符串"
    assert isinstance(_srcs, list) and len(_srcs) > 0, "sources 应为非空列表"
    assert all(isinstance(s, str) for s in _srcs), "每条来源应为字符串"

    # 历史截断验证
    _long_history = [
        {"role": "user" if i % 2 == 0 else "assistant", "content": f"轮{i//2+1}"}
        for i in range(20)
    ]
    _trunc = truncate_history(_long_history, max_turns=3)
    assert len(_trunc) == 6, f"3轮=6条，得{len(_trunc)}"
    assert _trunc[0]["content"] == "轮8", f"第一条应是第8轮，得{_trunc[0]['content']}"

    print("✅ podcast_qa 端到端验证通过")
    print(f"  答案: {_ans[:50]}")
    print(f"  来源: {_srcs}")
    print(f"  历史截断: 20条 → 6条 (max_turns=3)")
except (NotImplementedError, TypeError):
    print("⚠️ podcast_qa 尚未实现，完成5个TODO后重新运行")

## 6. 从"答有出处"到"自己动手"——ReAct Agent（工具调用）

前面的 `podcast_qa` 已经会"先查资料再回答"了。但它只会一条路：检索 → 生成。
遇到"谁弹吉他，顺便帮我算一下 6×7"这种既要**查**又要**算**的问题，纯 RAG 就卡住了——
知识库里没有"6×7"的答案，而 LLM 心算又常出错。

**Agent 的思路**：别让 LLM 一口气把答案说完，而是像人做题一样**边想边用工具**：

```
Thought:     我得先知道谁弹吉他。
Action:      search[who plays guitar]
Observation: Bob played guitar in the session.
Thought:     好，接下来算 6×7。
Action:      calculator[6*7]
Observation: 42
Thought:     两件事都有答案了。
Final Answer: 弹吉他的是 Bob；6×7=42。
```

这就是 **ReAct**（Reasoning + Acting）循环：**Thought（想）→ Action（调工具）→ Observation（看结果）**，
如此往复，直到 LLM 吐出 `Final Answer:` 才停。

**关键点：到底谁在转这个圈？**
- **LLM 只管"下一步做什么"**：读一遍当前的草稿纸（scratchpad），吐出一行 `Thought:` + 一个 `Action: 工具名[输入]`，或者宣布 `Final Answer:`。它**不**真的执行任何东西。
- **循环体（我们自己写的代码）才是真正的执行者**：把 `Action: 工具名[输入]` 解析出来，去调那个真正的 Python 函数，把返回值当作 `Observation:` 贴回草稿纸，再喂给 LLM 转下一圈。

**为什么不用 langchain？** Aurora 的铁律是 **no API wrappers**——ReAct 循环拆开看不过就是"正则解析 + 函数调度 + 字符串拼接"，二十行就能写透。框架只会把这层最该被理解的逻辑藏进黑箱。亲手写一遍，你才真的知道一个 Agent 在干什么。

**本节的两个工具（都复用课程既有代码，CPU、离线、可复现）**：
- `calculator(expr)`：从零实现的安全表达式求值，绝不联网。
- `search(query)`：直接把 **L89/L90 那套 from-scratch TF-IDF 检索器**包成一个工具——检索本身就是 Agent 的一种"感官"。


In [ ]:
import re

# ── 工具 1：计算器 —— 从零实现的安全求值，不用任何外部库 ─────────────────
def calculator(expr):
    """只允许数字与 + - * / ( ) . 空格 的安全表达式求值。"""
    allowed = set("0123456789+-*/(). ")
    if not set(expr) <= allowed:
        return "错误：表达式含非法字符"
    try:
        # __builtins__ 置空，杜绝任意代码执行（不是 API wrapper，是我们自己的沙箱）
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"错误：{e}"

# ── 工具 2：search —— 复用 L89/L90 的 from-scratch TF-IDF 检索器 ─────────
def search(query):
    """把 TF-IDF 检索包成一个 Agent 工具，返回最相关的一句话。"""
    hits = retrieve(query, synth_index, synth_chunks, _stub_embed, top_k=1)
    return hits[0]["text"] if hits else "无结果"

TOOLS = {"calculator": calculator, "search": search}

# ── 确定性 ReAct 规划器：脚本化的"假 LLM"，完全可复现、零网络 ────────────
class ScriptedReActLLM:
    """扮演 ReAct 里的 LLM：数一数草稿纸上已有几条 Observation，据此决定下一步。

    真实 LLM 靠语言理解来规划；这里用"数 Observation 条数"的确定性脚本代替，
    保证课堂结果每次一模一样。它吐出的字符串严格遵守 ReAct 文本协议：
      · 中间步：一行 `Thought:` + 一行 `Action: 工具名[输入]`
      · 收尾： `Final Answer: ...`
    """
    def generate(self, prompt):
        obs = re.findall(r"Observation:\s*(.*)", prompt)
        n = len(obs)
        if n == 0:
            return ("Thought: 我需要先查一下谁弹吉他。\n"
                    "Action: search[who plays guitar]")
        if n == 1:
            return ("Thought: 知道乐手了，接下来算一下 6*7。\n"
                    "Action: calculator[6*7]")
        # 已拿到两条观察，综合作答
        return f"Final Answer: 弹吉他的信息是「{obs[0]}」；6*7={obs[1]}。"

# 先各自单测一下两个工具与规划器的第一步（此刻草稿纸还没有 Observation）
print("calculator('6*7')      ->", calculator("6*7"))
print("search('who plays guitar') ->", search("who plays guitar"))
print("LLM 第一步：\n" + ScriptedReActLLM().generate("Question: demo\n"))


## 7. ✏️ 实现 ReAct 循环 `run_agent(question, llm, tools, max_steps=5)`

轮到你把"想—做—看"这个圈亲手转起来。循环体每一轮要做 5 件事：

1. 调 `llm.generate(scratchpad)` 拿到本步文本 `step`；
2. 把 `step` 追加进 `scratchpad`（和 `trace` 记录）；
3. 若 `step` 里出现 `Final Answer:`，取其后的文本作为 `answer`，返回 `(answer, trace)`；
4. 否则用正则解析 `Action: 工具名[输入]`，从 `tools` 取出对应函数并调用，得到 `observation`；
5. 把 `Observation: {observation}` 追加进 `scratchpad`（和 `trace`），进入下一轮。

超过 `max_steps` 还没拿到 `Final Answer:` 就应当报错（防死循环）。


In [ ]:
def run_agent(question, llm, tools, max_steps=5):
    """执行 ReAct 循环：Thought → Action → Observation → ... → Final Answer。

    Parameters
    ----------
    question  : str              用户问题
    llm       : object           带 .generate(prompt) -> str 的规划器
    tools     : dict[str, func]  工具名 → 可调用函数（输入一个 str，返回一个 str）
    max_steps : int              最多转几圈（防死循环）

    Returns
    -------
    (answer: str, trace: list[str])
        answer 是 `Final Answer:` 后面的文本；trace 是每一步（含 Observation）的文本记录。
    """
    scratchpad = f"Question: {question}\n"   # 草稿纸：不断追加 Thought/Action/Observation
    trace = []
    for _ in range(max_steps):
        # TODO 1：step = llm.generate(scratchpad)
        # TODO 2：把 step 追加进 scratchpad 和 trace
        # TODO 3：若 "Final Answer:" 在 step 中，取其后文本作为 answer，return (answer, trace)
        # TODO 4：否则用 re.search(r"Action:\s*(\w+)\[(.*?)\]", step) 解析工具名与输入，
        #          从 tools 取函数并调用，得到 observation
        # TODO 5：把 f"Observation: {observation}" 追加进 scratchpad 和 trace，进入下一轮
        raise NotImplementedError(
            "实现 ReAct 循环的一步：解析 Action → 调工具 → 追加 Observation")
    raise RuntimeError(f"超过 max_steps={max_steps} 仍未得到 Final Answer")


In [ ]:
# ✅ 验证：ReAct Agent 应在有限步内用两个工具得出正确答案
try:
    _answer, _trace = run_agent(
        "谁弹吉他？顺便算一下 6 乘 7。",
        ScriptedReActLLM(),
        TOOLS,
        max_steps=5,
    )
except (NotImplementedError, TypeError):
    print("⬜ 请先实现 run_agent() 的 ReAct 循环")
else:
    print("=== Agent 轨迹（scratchpad）===")
    for _line in _trace:
        print(_line)
    print("\n最终答案：", _answer)

    _n_obs = sum(1 for l in _trace if l.startswith("Observation:"))
    assert "Bob" in _answer, "search 工具应检索到 Bob 弹吉他"
    assert "42" in _answer, "calculator 工具应算出 6*7=42"
    assert _n_obs == 2, f"应恰好调用两次工具（两条 Observation），得 {_n_obs}"
    assert len(_trace) <= 2 * 5, "应在有限步内收敛，不应逼近 max_steps"
    print("\n✅ ReAct Agent 用 search + calculator 两个工具，在有限步内得出正确答案")


## 本课收束

本节实现了 `podcast_qa(question, rag_index, chunks, embed_fn, llm, history)` —— 它将检索到的 context chunks 拼入结构化 prompt，输出 `(answer: str, sources: list[str])` 两元组，让每条答案都可追溯到具体播客时间戳。`build_prompt`、`truncate_history`、`format_sources` 三个辅助函数共同构成 `aurora/podcast_intelligence/qa.py` 的完整骨架。

在此之上，本节还从零手写了一个 **ReAct 工具调用 Agent**：`run_agent()` 反复执行 **Thought → Action(工具) → Observation** 循环，直到 LLM 给出 `Final Answer:`。两个工具——从零实现的 `calculator` 与复用 TF-IDF 检索器的 `search`——让 Agent 既能算又能查，全程 CPU、离线、可复现，未借助任何 Agent 框架。下一课：**L91**。

In [ ]:
# ✏️ 本课自评
l90_review = {
    "rag_pipeline_intuition":    None,  # 理解对话式RAG=滑动窗口历史+检索+prompt组装+LLM生成？True/False
    "build_prompt_understood":   None,  # 理解prompt四层结构(SYSTEM/CONTEXT/HISTORY/QUESTION)？True/False
    "podcast_qa_impl":           None,  # podcast_qa()5步实现正确，端到端checker通过？True/False
    "source_attribution":        None,  # format_sources去重保序，理解Precision@k？True/False
    "whiteboard_passed":         None,  # 白板挑战5步数据流表格填写通关？True/False
    "react_agent_impl":          None,  # run_agent() ReAct循环实现正确，Agent用两个工具得出正确答案？True/False
}

unfilled = [k for k, v in l90_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l90_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L90 全部通关！进入 L91：注意力图解')

---

→ **下一课**　[L91 · 注意力图解](L91_visual_llm.ipynb)

> 下节课将学习 **注意力图解**：多头注意力权重热力图，LoRA 低秩结构可视化。